# Neural Fortran Examples: Convolutional Network on MNIST

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/cnn_mnist.f90`. It reshapes each digit into a one-channel $28 \times 28$ image, applies two convolutions and two max-pools, and finishes with a 10-way softmax. The library inserts a flatten layer where a dense layer follows a rank-3 map.

## The network

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist/app/cnn_mnist.f90
:language: fortran
:lines: 1-31
```

`load_mnist` is called with the optional test arrays, so this program loads the 10000 test images as well as the 50000 training and 10000 validation images.

`reshape(1, 28, 28)` is the layout `(channels, width, height)`. A convolution of kernel width 3 uses the valid-convolution size in `nf_conv2d_layer`,

$$
\left\lfloor\frac{n - 3}{s}\right\rfloor + 1
$$

with stride $s = 1$, and the constructor requires the two kernel sides to be equal. The spatial sizes implied by those formulas are:

| Layer | Output shape | Notes |
| --- | --- | --- |
| reshape | $1 \times 28 \times 28$ | one channel |
| conv, 8 filters, $3 \times 3$, ReLU | $8 \times 26 \times 26$ | $(28-3)+1 = 26$ |
| max-pool, $2 \times 2$, stride 2 | $8 \times 13 \times 13$ | integer division by the stride |
| conv, 16 filters, $3 \times 3$, ReLU | $16 \times 11 \times 11$ | $(13-3)+1 = 11$ |
| max-pool, $2 \times 2$, stride 2 | $16 \times 5 \times 5$ | $11 / 2 = 5$ |
| flatten, inserted by the constructor | 400 | $16 \cdot 5 \cdot 5$ |
| dense, softmax | 10 | $400 \cdot 10 + 10 = 4010$ parameters |

The source does not write a `flatten()` call. `network` adds one when a dense layer follows a convolution or a pool. `print_info` on this run prints the shapes in the table. A convolution kernel is shared, so the first layer has \(8 \cdot 1 \cdot 3 \cdot 3 + 8 = 80\) parameters and the second has \(16 \cdot 8 \cdot 3 \cdot 3 + 16 = 1168\). The dense layer has 4010. Those are the printed counts.


## The epoch loop and the accuracy

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist/app/cnn_mnist.f90
:language: fortran
:lines: 33-64
```

The loop runs 20 epochs. Each call to `train` uses one epoch, batch size 16, and SGD with learning rate 0.001. As in the dense program, a step is a random contiguous window, and `update` divides the gradient by the batch size. There are $50000 / 16 = 3125$ windows per epoch.

Accuracy is the same `maxloc` comparison as the dense example. The loop prints it on the validation images. After epoch 20 the program prints it once on the test images.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist/app/cnn_mnist.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `cnn_mnist` from the release build, run in a directory that already held the six MNIST files. Standard error was empty and the process exited 0.

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist/cnn_mnist.out
:language: text
```

## What the run did

![Training curve](images/nf_cnn_mnist.png)

The validation accuracy printed at the end of each epoch:

| Epoch | Validation accuracy (%) |
| ---: | ---: |
| 1 | 40.30 |
| 2 | 41.48 |
| 3 | 43.90 |
| 4 | 40.12 |
| 5 | 63.62 |
| 6 | 77.65 |
| 7 | 80.59 |
| 8 | 80.57 |
| 9 | 81.81 |
| 10 | 82.10 |
| 11 | 88.95 |
| 12 | 89.57 |
| 13 | 89.22 |
| 14 | 90.37 |
| 15 | 90.40 |
| 16 | 90.94 |
| 17 | 90.18 |
| 18 | 90.16 |
| 19 | 88.25 |
| 20 | 9.83 |

The test accuracy printed after the last epoch is 9.82 percent.


## Conclusion

The convolution program uses the same `train` and `maxloc` accuracy as the dense MNIST example. On this run the validation accuracy reaches 90.94 percent at epoch 16. Epoch 20 prints 9.83 percent, and the test accuracy measured after that epoch is 9.82 percent. The program does not keep the earlier weights. The saved run is the collapsed network, and the page reports that number.